# Base de modelado y evaluación — TP Ciencia de Datos

**Alcance:** selección de clasificadores, hiperparámetros, comparación mediante validación cruzada y evaluación final. **No ejecuta ningún modelo al abrirlo.**

**Origen:** adaptado de `evaluate_model()` de la Clase 5 y de ejemplos con `Pipeline`, `GridSearchCV`, métricas y clasificadores de las Clases 5, 6 y 7. Las funciones de orquestación que siguen son **adaptaciones nuestras**, no código literal del profesor.

**Contrato de integración:** el equipo debe proporcionar cuatro variables: `X_train`, `X_test`, `y_train`, `y_test`; `X_train` y `X_test` deben ser Series de textos preparados para el ejemplo de abajo. Si quieren incorporar características numéricas, deberán proporcionar un `preprocessor` válido y cambiar solamente la construcción de los pipelines. **No incluir jamás `polarity` dentro de X**.

**Orden:** fijar métrica → seleccionar/optimizar solo con entrenamiento → comparar modelos → diagnóstico opcional → elegir candidato → evaluación final del test reservado. No elegir el ganador por la puntuación del test.

## 0. Antes de integrar (acuerdos con el equipo)

- Conservar conjunto de prueba con **distribución original** (sin submuestreo previo).
- Aplicar normalización, vocabulario TF-IDF y cualquier transformación aprendida **dentro de CV**.
- Decidir cómo tratar textos duplicados idénticos que podrían cruzar train/test.
- Acordar si entregan textos o texto + variables numéricas. El ejemplo siguiente acepta **textos**.
- Verificar que el preprocesamiento acordado no elimine negaciones importantes para el sentimiento.
- Confirmar clase objetivo (`N`, `NEU`, `P`) y evitar que `polarity` entre entre las features.

## 1. Criterio de selección (definir antes de ejecutar)

**Propuesta inicial:** optimizar `f1_macro` porque `NEU` es mayoritaria y queremos valorar el desempeño en `N`, `NEU`, `P` por igual. Revisar particularmente el **recall de N y P** y las confusiones hacia `NEU` para detectar falsos neutrales. Si el equipo argumenta que alguna confusión es más costosa, documentar esa prioridad antes de seleccionar modelos.

**Nota:** el Macro F1 es un promedio por clase, no una medida específica del costo de confundir `N` con `P` o con `NEU`; esas diferencias deben verse en la matriz de confusión.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, learning_curve, validation_curve
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay


## 2. Punto de integración

No ejecutés la siguiente celda hasta que tu equipo tenga listas las variables. **Para esta primera versión asumimos `X` textual**, como en los notebooks de NLP. Separación de datos y manejo de duplicados quedan a cargo de la integración.


In [ ]:
# Deben estar definidas ANTES de entrenar:
# X_train  -> Series de textos para entrenar
# X_test   -> Series de textos reservados
# y_train  -> etiquetas de entrenamiento (N, NEU, P)
# y_test   -> etiquetas reservadas
#
# Ejemplo de comprobación cuando el equipo entregue los conjuntos:
# print(len(X_train), len(X_test))
# print(y_train.value_counts())
# print(y_test.value_counts())


## 3. Modelos candidatos — versión texto + TF-IDF

Se eligieron modelos que aceptan matrices dispersas de TF-IDF. `GaussianNB` **no acepta directamente matrices dispersas**; para incluirlo hace falta decidir una representación densa (y evaluar su costo de memoria) o usar features numéricas apropiadas. `AdaBoostClassifier` y `GradientBoostingClassifier` quedan como extensiones opcionales que deben revisarse por eficiencia con TF-IDF y atribuirse al bloque de ensambles; no hace falta probar todo para justificar el TP.


In [ ]:
candidatos = {
    "Logistic Regression": (
        Pipeline([("tfidf", TfidfVectorizer()),
                  ("classifier", LogisticRegression(max_iter=1000, random_state=42))]),
        {"classifier__C": [0.01, 0.1, 1, 10, 100]}
    ),
    "Multinomial Naive Bayes": (
        Pipeline([("tfidf", TfidfVectorizer()),
                  ("classifier", MultinomialNB())]),
        {"classifier__alpha": [0.01, 0.1, 1, 10, 100]}
    ),
    "Decision Tree": (
        Pipeline([("tfidf", TfidfVectorizer()),
                  ("classifier", DecisionTreeClassifier(random_state=42))]),
        {"classifier__max_depth": [3, 5, 10, None]}
    ),
    "Random Forest": (
        Pipeline([("tfidf", TfidfVectorizer()),
                  ("classifier", RandomForestClassifier(random_state=42))]),
        {"classifier__n_estimators": [50, 100],
         "classifier__max_depth": [5, 10, None]}
    ),
    "SVM lineal": (
        Pipeline([("tfidf", TfidfVectorizer()),
                  ("classifier", SVC(kernel="linear"))]),
        {"classifier__C": [0.01, 0.1, 1, 10, 100]}
    ),
}


## 4. Selección y tabla comparativa — no utiliza X_test

`GridSearchCV` hace CV y reajusta automáticamente el Pipeline ganador sobre todo `X_train`. La desviación estándar de los folds ayuda a ver estabilidad descriptiva; **no es una prueba de significación**. Para comparar modelos de forma más controlada, se usa el mismo `cv` y se puede mantener explícitamente las mismas particiones estratificadas.

In [ ]:
def seleccionar_modelo(nombre, pipeline, param_grid, X_train, y_train):
    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1,
        return_train_score=True
    )
    grid.fit(X_train, y_train)
    idx = grid.best_index_
    resultado = {
        "Modelo": nombre,
        "Macro F1 CV": grid.best_score_,
        "Desvío entre folds": grid.cv_results_["std_test_score"][idx],
        "Hiperparámetros": grid.best_params_
    }
    return grid, resultado

# Ejecutar SOLO después de conectar X_train e y_train:
# grids = {}
# filas = []
# for nombre, (pipeline, parametros) in candidatos.items():
#     grid, fila = seleccionar_modelo(nombre, pipeline, parametros, X_train, y_train)
#     grids[nombre] = grid
#     filas.append(fila)
# comparacion = pd.DataFrame(filas).sort_values("Macro F1 CV", ascending=False)
# display(comparacion)


## 5. Diagnóstico — curvas del profesor adaptadas a Macro F1 y train solamente

No hace falta calcular todas las curvas para todos los candidatos si son costosas; se pueden priorizar finalistas. Las curvas usan datos ya empleados en la selección, por lo que **no constituyen evaluación independiente**.

In [ ]:
def graficar_learning_curve(nombre, modelo, X_train, y_train):
    tamanios, train_scores, val_scores = learning_curve(
        modelo, X_train, y_train, cv=5, scoring="f1_macro", n_jobs=-1,
        train_sizes=np.linspace(0.2, 1.0, 5)
    )
    plt.figure(figsize=(8, 5))
    plt.plot(tamanios, train_scores.mean(axis=1), marker="o", label="Entrenamiento")
    plt.plot(tamanios, val_scores.mean(axis=1), marker="o", label="Validación")
    plt.xlabel("Cantidad de ejemplos de entrenamiento")
    plt.ylabel("Macro F1")
    plt.title(f"Learning Curve — {nombre}")
    plt.legend()
    plt.show()


def graficar_validation_curve(nombre, modelo, X_train, y_train,
                              parametro, valores, escala_log=False):
    train_scores, val_scores = validation_curve(
        modelo, X_train, y_train, param_name=parametro,
        param_range=valores, cv=5, scoring="f1_macro", n_jobs=-1
    )
    plt.figure(figsize=(8, 5))
    plt.plot(valores, train_scores.mean(axis=1), marker="o", label="Entrenamiento")
    plt.plot(valores, val_scores.mean(axis=1), marker="o", label="Validación")
    if escala_log:
        plt.xscale("log")
    plt.xlabel(parametro.split("__")[-1])
    plt.ylabel("Macro F1")
    plt.title(f"Validation Curve — {nombre}")
    plt.legend()
    plt.show()

# Ejemplo, DESPUÉS de crear grids:
# finalista = "Logistic Regression"
# graficar_learning_curve(finalista, grids[finalista].best_estimator_, X_train, y_train)
# graficar_validation_curve(finalista, grids[finalista].best_estimator_,
#                           X_train, y_train, "classifier__C", [0.01, 0.1, 1, 10, 100], True)

# aca se podria adaptar la funcion para que me compare a todos los modelos en el mismo grafico, o no tiene sentido?


## 5B. Diagnóstico unificado de los modelos (solo validación cruzada)

**Inspirado en el `evaluate_model()` de Clase 5:** mantiene matriz de confusión, `classification_report`, `learning_curve` y `validation_curve`, pero evita utilizar `X_test` durante selección. Se incluyen dos gráficos complementarios: puntuaciones de cada fold y distribución real frente a predicha.

**NUESTRA ADAPTACIÓN — distribución de sentimientos:** computamos `value_counts(normalize=True)` en las etiquetas reales y en las predicciones fuera de fold (OOF), y la diferencia absoluta de proporciones. **No es una métrica proporcionada por la cátedra.** Para obtener esas predicciones se usa `cross_val_predict`, un auxiliar de Scikit-Learn no identificado en los ejemplos consultados de la cátedra. Su papel es repetir el mismo esquema de CV y producir una predicción por texto, cuando ese texto fue validación. Si la consigna exige exclusivamente funciones vistas en clase, consultar al profesor o reemplazar este bloque por una separación explícita de validación.

**Precaución metodológica:** las configuraciones `best_estimator_` ya se seleccionaron usando `X_train`; por eso estos gráficos son *diagnósticos exploratorios de selección*, no una estimación independiente. Solo `evaluar_modelo_final()` debe recibir `X_test`. La distribución predicha puede aproximarse a la real aun con predicciones individuales erróneas.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

def evaluar_modelos_cv(grids, configuraciones, X_train, y_train):
    """Diagnósticos de selección; NO recibe X_test ni y_test."""
    resumen = []
    for nombre, grid in grids.items():
        modelo = grid.best_estimator_
        config = configuraciones.get(nombre, {})
        # NUESTRA ADAPTACIÓN: predicciones fuera de fold para cada texto de entrenamiento.
        y_pred_cv = cross_val_predict(modelo, X_train, y_train, cv=5, n_jobs=-1)
        clases = sorted(pd.Series(y_train).unique())

        reales = pd.Series(y_train).value_counts(normalize=True).reindex(clases, fill_value=0)
        predichas = pd.Series(y_pred_cv).value_counts(normalize=True).reindex(clases, fill_value=0)
        diferencia = predichas - reales  # positivo = sobreestimación de la clase
        error_promedio_pp = diferencia.abs().mean() * 100
        # FIN DE NUESTRA ADAPTACIÓN de distribuciones.

        reporte = classification_report(y_train, y_pred_cv, output_dict=True, zero_division=0)
        train_sizes, train_scores, val_scores = learning_curve(
            modelo, X_train, y_train, cv=5, scoring='f1_macro', n_jobs=-1,
            train_sizes=np.linspace(0.2, 1.0, 5)
        )
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f'{nombre} — diagnóstico en validación (sin test)', fontsize=15)

        # 1. Matriz de confusión OOF (el gráfico original de la cátedra, en multiclase)
        cm = confusion_matrix(y_train, y_pred_cv, labels=clases)
        axes[0, 0].imshow(cm, cmap='Blues')
        for i in range(len(clases)):
            for j in range(len(clases)):
                axes[0, 0].text(j, i, cm[i, j], ha='center', va='center')
        axes[0, 0].set(xticks=range(len(clases)), yticks=range(len(clases)),
                       xticklabels=clases, yticklabels=clases,
                       xlabel='Predicho', ylabel='Real', title='1. Matriz de confusión CV')
        axes[0, 0].tick_params(axis='x', labelrotation=30)

        # 2. F1 por clase del classification_report: adaptación de visualización
        f1_clases = [reporte[str(c)]['f1-score'] for c in clases]
        axes[0, 1].bar([str(c) for c in clases], f1_clases)
        axes[0, 1].set(title='2. F1 por clase (CV)', ylabel='F1', ylim=(0, 1.05))
        axes[0, 1].tick_params(axis='x', labelrotation=30)

        # 3. NUESTRA ADAPTACIÓN: distribución real/predicha (porcentajes)
        x = np.arange(len(clases))
        axes[0, 2].bar(x - 0.2, reales.values * 100, width=0.4, label='Real')
        axes[0, 2].bar(x + 0.2, predichas.values * 100, width=0.4, label='Predicha')
        axes[0, 2].set(xticks=x, xticklabels=clases,
                       title=f'3. Distribución CV (error medio: {error_promedio_pp:.1f} pp)',
                       ylabel='Porcentaje de textos')
        axes[0, 2].tick_params(axis='x', labelrotation=30)
        axes[0, 2].legend()
        # FIN DE NUESTRA ADAPTACIÓN de distribuciones.

        # 4. Learning curve (profesor, cambiando a X_train y Macro F1)
        axes[1, 0].plot(train_sizes, train_scores.mean(axis=1), marker='o', label='Train')
        axes[1, 0].plot(train_sizes, val_scores.mean(axis=1), marker='o', label='Validación')
        axes[1, 0].set(title='4. Learning Curve', xlabel='Textos de entrenamiento', ylabel='Macro F1', ylim=(0,1.05))
        axes[1, 0].legend()

        # 5. Validation curve (profesor, solo si la configuración lo especifica)
        if 'parametro' in config and 'valores' in config:
            tc, vc = validation_curve(
                modelo, X_train, y_train, param_name=config['parametro'],
                param_range=config['valores'], cv=5, scoring='f1_macro', n_jobs=-1
            )
            axes[1, 1].plot(config['valores'], tc.mean(axis=1), marker='o', label='Train')
            axes[1, 1].plot(config['valores'], vc.mean(axis=1), marker='o', label='Validación')
            if config.get('escala_log', False):
                axes[1, 1].set_xscale('log')
            axes[1, 1].set(title='5. Validation Curve', xlabel=config['parametro'], ylabel='Macro F1', ylim=(0,1.05))
            axes[1, 1].legend()
        else:
            axes[1, 1].set_title('5. Sin hiperparámetro para curva')
            axes[1, 1].axis('off')

        # 6. Puntaje en cada fold del GridSearch (diagnóstico de variabilidad)
        indice = grid.best_index_
        valores_fold = [grid.cv_results_[f'split{i}_test_score'][indice] for i in range(grid.n_splits_)]
        axes[1, 2].bar(range(1, len(valores_fold) + 1), valores_fold)
        axes[1, 2].set(title='6. Macro F1 por fold del GridSearch', xlabel='Fold', ylabel='Macro F1', ylim=(0,1.05))

        plt.tight_layout()
        plt.show()
        print('Modelo:', nombre)
        print('Mejores hiperparámetros:', grid.best_params_)
        print('Macro F1 CV (selección):', round(grid.best_score_, 4))
        print('Diferencia media de distribución:', round(error_promedio_pp, 2), 'pp')
        print('Reporte de predicciones de CV (diagnóstico):')
        print(classification_report(y_train, y_pred_cv, zero_division=0))
        resumen.append({
            'Modelo': nombre,
            'Macro F1 CV (GridSearch)': grid.best_score_,
            'Error medio distribución (pp)': error_promedio_pp,
            'Mejores parámetros': grid.best_params_
        })
    return pd.DataFrame(resumen).sort_values('Macro F1 CV (GridSearch)', ascending=False)


### Uso con los grids que ya entrenaste

No es necesario volver a ejecutar la búsqueda si ya existen `grid_lr`, `grid_nb`, `grid_svm`, `grid_rf`. Esta función **sí entrena modelos adicionales** internamente para las curvas y para obtener predicciones fuera de fold. Los resultados de `cross_val_predict` son diagnósticos con la configuración ya elegida; pueden diferir del `best_score_` del GridSearch.

**El gráfico 3 es nuestra contribución original.** Mostramos el promedio de las diferencias absolutas de porcentajes entre las tres clases (en puntos porcentuales), que es un criterio complementario y no una métrica de la cátedra. Antes de utilizarlo para seleccionar modelos, decidir y documentar cómo se ponderará frente a Macro F1.

In [ ]:
# Ejecutar cuando ya estén definidos y entrenados estos cuatro grids.
# grids = {
#     'Logistic Regression': grid_lr,
#     'Naive Bayes': grid_nb,
#     'SVM': grid_svm,
#     'Random Forest': grid_rf
# }
# configuraciones = {
#     'Logistic Regression': {'parametro':'classifier__C','valores':[0.01,0.1,1,10,100],'escala_log':True},
#     'Naive Bayes': {'parametro':'classifier__alpha','valores':[0.01,0.1,1,10,100],'escala_log':True},
#     'SVM': {'parametro':'classifier__C','valores':[0.01,0.1,1,10,100],'escala_log':True},
#     'Random Forest': {'parametro':'classifier__n_estimators','valores':[50,100,150,200]}
# }
# tabla_comparativa = evaluar_modelos_cv(grids, configuraciones, X_train, y_train)
# tabla_comparativa


## 6. Elección documentada — ANTES de mirar test

Registrar el modelo ganador según el criterio definido y, si hay puntuaciones muy próximas, considerar estabilidad entre folds, complejidad y costo de entrenamiento. La decisión debe quedar fija antes de la siguiente sección. No ajustar nuevamente el ganador tras mirar el test.

In [ ]:
# Elegir un nombre entre los presentes en grids, luego de comparar CV:
# nombre_elegido = "Logistic Regression"
# modelo_elegido = grids[nombre_elegido].best_estimator_


## 7. Evaluación FINAL — ejecutar una vez con el modelo elegido

Este bloque **no es parte de la búsqueda**. Usa exclusivamente `X_test` e `y_test` para informar desempeño final. El reporte es multiclase; no utiliza el ROC binario ni la elección de umbrales que estaban en la función original del profesor.

In [ ]:
def evaluar_modelo_final(nombre, modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    print(f"Modelo final: {nombre}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, labels=["N", "NEU", "P"], cmap="Blues"
    )
    plt.title(f"Matriz de confusión — {nombre}")
    plt.show()
    return y_pred

# Ejecutar AL FINAL, cuando la selección esté cerrada:
# predicciones = evaluar_modelo_final(nombre_elegido, modelo_elegido, X_test, y_test)


## 8. Guía de interpretación y conclusiones

En la entrega, justificar: **(1)** qué errores perjudican más el objetivo del problema (ej. confundir `N`/`P` como `NEU`); **(2)** por qué la métrica primaria representa ese criterio; **(3)** qué modelos e hiperparámetros se compararon y por qué; **(4)** comparación CV y variabilidad; **(5)** evidencia de sobreajuste/subajuste donde corresponda; **(6)** resultados finales y errores por clase; **(7)** limitaciones y mejoras posibles.

**Coordinación:** pedir al equipo esquema exacto de X, fecha de congelación del conjunto de prueba, protocolo para duplicados y balanceo, y confirmación de que TF-IDF/escaladores se ajustan solamente con entrenamiento en cada fold.

**Referencia:** el `evaluate_model` del profesor de Clase 5 reúne GridSearchCV, accuracy, matriz de confusión, ROC y curvas. Lo separamos deliberadamente para evitar utilizar test durante selección, adaptarlo a multiclase y poder comparar varios clasificadores.
